# EICC GRPO Training Notebook

Colab-ready notebook for the Enterprise Incident Command Center:
1. Clone repo & install dependencies
2. Verify environment works (dry-run)
3. Run baseline evaluation
4. Train with GRPO
5. Run post-training evaluation
6. Plot reward curves and inspect artifacts

In [ ]:
# Step 1: Clone your repo
!git clone https://github.com/anujkamaljain/OpenEnv-Customer-Support.git
%cd OpenEnv-Customer-Support

In [ ]:
# Step 2: Install training dependencies (DGX-stable, wheel-only path)
# Run this once, then restart kernel and rerun from Step 1.
%pip install -q -U pip setuptools wheel jedi

# Clean potentially broken partial installs from earlier attempts.
!python -m pip uninstall -y pyarrow xformers unsloth unsloth_zoo datasets mergekit sentencepiece llm-blender charset-normalizer cchardet aiohttp requests bitsandbytes

# Use conda binaries for packages that often fail to compile from source.
!conda install -y -c conda-forge "pyarrow=18.1.0" "datasets=2.21.*" "sentencepiece>=0.1.99"

# Pin torch to the newest version available on this DGX index.
%pip install -q -U "torch==2.6.*"

# Install core stack with wheels only (prevents accidental source builds).
%pip install -q -U --only-binary=:all: "trl==0.15.2" "transformers==4.51.3" peft matplotlib accelerate

# Repair HTTP/text deps used by datasets/aiohttp (fixes broken charset_normalizer installs).
%pip install -q -U --force-reinstall --no-cache-dir --only-binary=:all: "charset-normalizer==3.4.2" "aiohttp==3.10.11" "requests==2.32.3"

# Keep bitsandbytes absent on this index and install unsloth without transitive re-resolution.
!python -m pip uninstall -y bitsandbytes
%pip install -q -U unsloth_zoo --no-deps
%pip install -q -U unsloth --no-deps

print("Install step done. Restart kernel now, then rerun from Step 1.")

In [ ]:
# Step 3: Verify training stack + environment (quick sanity check)
import os
import importlib.metadata as im

# DGX environments may not have build toolchains for xformers; force fallback path.
os.environ.setdefault("UNSLOTH_DISABLE_XFORMERS", "1")

# Keep transformers from importing bitsandbytes integration on constrained DGX images.
try:
    import transformers.utils.import_utils as _import_utils
    _import_utils.is_bitsandbytes_available = lambda *args, **kwargs: False
except Exception:
    pass

from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

print("unsloth:", im.version("unsloth"))
print("trl:", im.version("trl"))
print("transformers:", im.version("transformers"))

!python train.py --iterations 1 --episodes 1 --k 2 --dry-run

In [ ]:
# Step 4: Run baseline evaluation (before training)
!python evaluate.py --policy baseline --episodes-per-difficulty 5 --output-dir artifacts/eval

In [ ]:
# Step 5: GRPO Training (~6-8 hours on T4, ~2-3 hours on V100)
# Reduce iterations/episodes if short on time:
#   --iterations 10 --episodes 15 --k 2  (~2-3 hours on T4)
!python train.py --iterations 20 --episodes 30 --k 4 --eval-episodes 5 --output-dir artifacts/train

In [ ]:
# Step 6: Run post-training comparison evaluation
!python evaluate.py --policy compare --episodes-per-difficulty 5 --plot --output-dir artifacts/eval

In [ ]:
# Step 7: View results
from pathlib import Path
import json

baseline_path = Path("artifacts/eval/baseline_report.json")
trained_path = Path("artifacts/eval/trained_report.json")
if baseline_path.exists() and trained_path.exists():
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    trained = json.loads(trained_path.read_text(encoding="utf-8"))
    print("Baseline avg_normalized_reward:", baseline["avg_normalized_reward"])
    print("Trained avg_normalized_reward:", trained["avg_normalized_reward"])
    print("Behavior examples:")
    for line in trained.get("behavior_examples", []):
        print("-", line)
else:
    print("Run evaluation cells first.")

In [ ]:
# Step 8: Display reward curve
from pathlib import Path
from IPython.display import Image, display

curve = Path("artifacts/eval/reward_curves.png")
if curve.exists():
    display(Image(filename=str(curve)))
else:
    print("Run compare evaluation with --plot first.")